# 01. Engenharia de Dados & Geração de Pipeline

Este notebook é responsável pelo download das ortofoto(s) GeoTIFF, fatiamento em blocos de 640x640, conversão cromática e geração do arquivo HDF5 inicial compilado com as imagens brutas.

**Responsáveis:** Pessoa 1 (Luidgi Varela Carneiro) e Pessoa 2 (Rafael de Lima Pereira)

In [ ]:
import sys
sys.path.append('..')
import utils
import rasterio
import numpy as np
import cv2

## 1. Download das Ortofotos GeoTIFF (Geoportal DF)

In [ ]:
# Código para download automático

## 2. Fatiamento em 640x640 (Pessoa 1)

In [ ]:
# Chamada de slice_geotiff

## 3. Conversão Cromática (RGB -> BGR) e Compilação HDF5 (Pessoa 2)

In [ ]:
# 3. Conversão Cromática (RGB -> BGR) e Compilação HDF5 (Pessoa 2)
from pathlib import Path

# Pasta com os tiles .tif gerados pelo fatiamento da Pessoa 1 (slice_geotiff).
# Ajuste para a subpasta da ortofoto processada (ex.: data/tiles/asa_norte_setor_01).
TILES_DIR = Path("../data/tiles")

# Caminho VERSIONADO do HDF5 no Google Drive.
# IMPORTANTE: nunca sobrescreva versões anteriores — use sufixos (_v1_raw, _v2_QA, ...).
HDF5_PATH = "dataset_v1_raw.h5"  # ex.: /content/drive/MyDrive/.../02_Datasets_HDF5/dataset_v1_raw.h5

# Coleta os caminhos dos tiles. Se você guardou o retorno de slice_geotiff em
# `generated_tiles`, prefira reutilizá-lo: tile_paths = [str(p) for p in generated_tiles]
tile_paths = sorted(str(p) for p in TILES_DIR.rglob("*.tif"))
print(f"Tiles .tif encontrados: {len(tile_paths)}")

# criar_hdf5_bruto chama carregar_tile internamente, que:
#   1. lê cada tile GeoTIFF com rasterio (RGB, formato CHW);
#   2. reordena os canais RGB -> BGR (ordem nativa do OpenCV) e garante uint8;
#   3. grava no grupo images/ com compressão gzip e inicializa labels/ vazio.
utils.criar_hdf5_bruto(HDF5_PATH, tile_paths)
print(f"HDF5 inicial criado em: {HDF5_PATH}")


In [ ]:
# Verificação rápida do HDF5 gerado (estrutura, dimensões e sanidade cromática)
import h5py
import matplotlib.pyplot as plt

with h5py.File(HDF5_PATH, "r") as f:
    nomes = sorted(f["images"].keys())
    print(f"Total de imagens no HDF5: {len(nomes)}")
    if nomes:
        amostra = f["images"][nomes[0]]
        print(f"Exemplo {nomes[0]}: shape={amostra.shape}, dtype={amostra.dtype}, "
              f"compression={amostra.compression}, chunks={amostra.chunks}")

        # As imagens estão em BGR; matplotlib espera RGB, então invertemos os canais.
        img_bgr = amostra[:]
        plt.imshow(img_bgr[..., ::-1])
        plt.title(f"{nomes[0]} (cores convertidas BGR->RGB para exibição)")
        plt.axis("off")
        plt.show()
